In [ ]:
# %% [markdown]
# # Stock Clustering & Mean-Reversion Trading Framework
# 
# This notebook implements a comprehensive trading strategy that:
# 1. Fetches stock data from MongoDB
# 2. Performs hierarchical clustering on correlated stocks
# 3. Generates mean-reversion signals within clusters
# 4. Includes enhanced risk management and backtesting
# 
# ## Improvements Over Basic Implementation:
# - Data preprocessing and outlier handling
# - Multiple clustering methods
# - Advanced signal generation with multiple timeframes
# - Risk management with position sizing
# - Realistic backtesting with transaction costs
# - Walk-forward optimization
# - Performance analytics

# %% [markdown]
# ## 1. Install Required Packages
# Uncomment if running for the first time

# %%
#!pip install pymongo pandas numpy matplotlib seaborn scipy scikit-learn statsmodels plotly ta
!pip install vectorbt

In [5]:
# %% [markdown]
# ## 2. Imports and Configuration

# %%
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import json
from typing import Dict, List, Tuple, Optional
import logging

# Database
from pymongo import MongoClient

# Clustering
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform, pdist
from sklearn.cluster import DBSCAN, OPTICS, SpectralClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Statistical Tests
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.vector_ar.vecm import coint_johansen

# Technical Analysis
import ta

# Plotting
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Backtesting
import vectorbt as vbt

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [6]:
import os
# %% [markdown]
# ## 3. Configuration Parameters

# %%
class Config:
    """Configuration parameters for the trading strategy"""
    
    # Data parameters
    TICKER_LIMIT = 50
    START_DATE = '2010-01-01'
    END_DATE = '2024-01-01'
    
    # Clustering parameters
    CLUSTER_METHOD = 'hierarchical'  # Options: hierarchical, optics, spectral, kmeans
    CLUSTER_THRESHOLD = 0.5
    MIN_CLUSTER_SIZE = 3
    MAX_CLUSTER_SIZE = 15
    
    # Signal parameters
    ZSCORE_ENTRY_LONG = -1.0
    ZSCORE_EXIT_LONG = -0.25
    ZSCORE_ENTRY_SHORT = 1.0
    ZSCORE_EXIT_SHORT = 0.25
    LOOKBACK_WINDOWS = [10, 20, 60]  # Multiple timeframes
    
    # Risk parameters
    MAX_POSITION_SIZE = 0.1  # 10% per position
    MAX_PORTFOLIO_EXPOSURE = 1.0  # 100% gross exposure
    TARGET_VOLATILITY = 0.20  # 20% annualized
    MAX_DRAWDOWN = 0.20  # 20% max drawdown
    TRANSACTION_COST = 0.001  # 0.1% per trade
    
    # Backtest parameters
    INITIAL_CAPITAL = 1000000
    REBALANCE_FREQ = 'B'  # Business daily
    
    # MongoDB configuration
    MONGO_URI = os.environ["MONGODB_URI"]
    DATABASE_NAME = "sse_stock_db"
    COLLECTION_NAME = "daily_data"

config = Config()

In [7]:
# %% [markdown]
# ## 4. Data Fetcher Class

# %%
class DataFetcher:
    """Handles data fetching from MongoDB and preprocessing"""
    
    def __init__(self, config):
        self.config = config
        self.client = None
        self.data = None
        
    def connect(self):
        """Connect to MongoDB"""
        try:
            self.client = MongoClient(self.config.MONGO_URI)
            logger.info("Connected to MongoDB successfully")
            return True
        except Exception as e:
            logger.error(f"Failed to connect to MongoDB: {e}")
            return False
    
    def fetch_data(self, tickers: List[str] = None) -> pd.DataFrame:
        """Fetch and preprocess stock data"""
        if not self.connect():
            raise ConnectionError("Cannot connect to MongoDB")
        
        db = self.client[self.config.DATABASE_NAME]
        collection = db[self.config.COLLECTION_NAME]
        
        # Get tickers if not provided
        if tickers is None:
            tickers = collection.distinct("ticker")[:self.config.TICKER_LIMIT]
            logger.info(f"Fetched {len(tickers)} tickers")
        
        dfs = []
        for t in tickers:
            try:
                # Fetch data with date filter
                query = {
                    "ticker": t,
                    "date": {
                        "$gte": {"$date": self.config.START_DATE + "T00:00:00Z"},
                        "$lte": {"$date": self.config.END_DATE + "T23:59:59Z"}
                    }
                }
                
                docs = list(collection.find(
                    query, 
                    {"_id": 0, "date": 1, "close": 1, "volume": 1}
                ))
                
                if len(docs) < 100:  # Skip if insufficient data
                    continue
                
                # Process MongoDB date format
                for d in docs:
                    if isinstance(d["date"], dict) and "$date" in d["date"]:
                        d["date"] = pd.to_datetime(d["date"]["$date"])
                
                # Convert to DataFrame
                df = pd.DataFrame(docs)
                df["date"] = pd.to_datetime(df["date"])
                df = df.set_index("date")
                
                # Handle missing values
                df = df[["close", "volume"]].rename(columns={"close": t, "volume": f"{t}_volume"})
                df = df.sort_index()
                
                # Forward fill for missing days, then drop remaining NaN
                df = df.resample('B').ffill().dropna()
                
                dfs.append(df)
                
            except Exception as e:
                logger.warning(f"Error processing {t}: {e}")
                continue
        
        if not dfs:
            raise ValueError("No valid data fetched")
        
        # Merge all tickers
        close_prices = pd.concat([df.iloc[:, 0] for df in dfs], axis=1)
        volumes = pd.concat([df.iloc[:, 1] for df in dfs], axis=1)
        
        # Align indices and fill missing values
        close_prices = close_prices.fillna(method='ffill').dropna()
        volumes = volumes.reindex(close_prices.index).fillna(method='ffill')
        
        self.data = {
            'prices': close_prices,
            'volumes': volumes,
            'returns': close_prices.pct_change().dropna()
        }
        
        logger.info(f"Data fetched successfully: {close_prices.shape[1]} tickers, {close_prices.shape[0]} periods")
        return self.data
    
    def preprocess_data(self) -> Dict:
        """Advanced data preprocessing"""
        if self.data is None:
            raise ValueError("Data not fetched. Call fetch_data() first.")
        
        returns = self.data['returns'].copy()
        prices = self.data['prices'].copy()
        
        # 1. Remove non-stationary series
        stationary_tickers = []
        for ticker in returns.columns:
            p_value = adfuller(returns[ticker].dropna())[1]
            if p_value < 0.05:  # 95% confidence of stationarity
                stationary_tickers.append(ticker)
            else:
                logger.debug(f"Dropping {ticker} - non-stationary (p={p_value:.3f})")
        
        returns = returns[stationary_tickers]
        prices = prices[stationary_tickers]
        
        # 2. Winsorize outliers (clip extreme returns)
        lower_quantile = returns.quantile(0.01)
        upper_quantile = returns.quantile(0.99)
        returns = returns.clip(lower=lower_quantile, upper=upper_quantile, axis=1)
        
        # 3. Calculate additional features
        volatilities = returns.rolling(20).std() * np.sqrt(252)
        
        # 4. Calculate technical indicators
        technical_features = {}
        for ticker in prices.columns:
            df_ta = pd.DataFrame(index=prices.index)
            df_ta['price'] = prices[ticker]
            df_ta['volume'] = self.data['volumes'][f"{ticker}_volume"]
            
            # RSI
            df_ta['rsi'] = ta.momentum.RSIIndicator(df_ta['price'], window=14).rsi()
            
            # MACD
            macd = ta.trend.MACD(df_ta['price'])
            df_ta['macd'] = macd.macd()
            df_ta['macd_signal'] = macd.macd_signal()
            
            # Bollinger Bands
            bb = ta.volatility.BollingerBands(df_ta['price'], window=20)
            df_ta['bb_high'] = bb.bollinger_hband()
            df_ta['bb_low'] = bb.bollinger_lband()
            df_ta['bb_width'] = (df_ta['bb_high'] - df_ta['bb_low']) / df_ta['price']
            
            technical_features[ticker] = df_ta
        
        self.data.update({
            'returns': returns,
            'prices': prices,
            'volatilities': volatilities,
            'technical_features': technical_features
        })
        
        logger.info(f"Data preprocessing complete. {len(stationary_tickers)} stationary tickers.")
        return self.data


In [8]:
# %% [markdown]
# ## 5. Stock Clustering Engine

# %%
class StockClustering:
    """Performs clustering on stocks based on return correlations"""
    
    def __init__(self, config):
        self.config = config
        self.cluster_labels = None
        self.cluster_metrics = {}
        
    def calculate_distance_matrix(self, returns: pd.DataFrame) -> pd.DataFrame:
        """Calculate distance matrix from correlation"""
        # Correlation matrix
        corr_matrix = returns.corr()
        
        # Handle NaN values
        corr_matrix = corr_matrix.fillna(0)
        
        # Convert to distance (1 - |correlation|)
        distance = 1 - corr_matrix.abs()
        
        # Ensure positive semi-definite
        distance.values[np.diag_indices_from(distance)] = 0
        
        return distance, corr_matrix
    
    def hierarchical_clustering(self, distance: pd.DataFrame) -> np.ndarray:
        """Perform hierarchical clustering"""
        condensed_dist = squareform(distance.values, checks=False)
        
        # Try different linkage methods
        linkage_methods = ['ward', 'average', 'complete', 'single']
        best_score = -1
        best_labels = None
        
        for method in linkage_methods:
            try:
                linked = linkage(condensed_dist, method=method)
                
                # Find optimal threshold using silhouette score
                thresholds = np.arange(0.3, 0.8, 0.1)
                for t in thresholds:
                    labels = fcluster(linked, t=t, criterion='distance')
                    
                    if len(set(labels)) > 1:
                        score = silhouette_score(distance.values, labels, metric='precomputed')
                        if score > best_score:
                            best_score = score
                            best_labels = labels
                            self.cluster_metrics['threshold'] = t
                            self.cluster_metrics['linkage_method'] = method
            except:
                continue
        
        if best_labels is None:
            # Default clustering
            linked = linkage(condensed_dist, method='average')
            best_labels = fcluster(linked, t=self.config.CLUSTER_THRESHOLD, criterion='distance')
        
        self.cluster_metrics['silhouette_score'] = best_score
        return best_labels
    
    def optics_clustering(self, distance: pd.DataFrame) -> np.ndarray:
        """Perform OPTICS clustering"""
        optics = OPTICS(
            min_samples=3,
            metric='precomputed',
            cluster_method='xi'
        )
        labels = optics.fit_predict(distance.values)
        
        # Handle noise points (-1 label)
        if -1 in labels:
            # Assign noise points to nearest cluster
            unique_labels = set(labels) - {-1}
            if unique_labels:
                for i, label in enumerate(labels):
                    if label == -1:
                        # Find nearest non-noise point
                        distances = distance.iloc[i]
                        valid_indices = [j for j, l in enumerate(labels) if l != -1]
                        if valid_indices:
                            nearest_idx = distances.iloc[valid_indices].idxmin()
                            labels[i] = labels[nearest_idx]
        
        return labels
    
    def spectral_clustering(self, distance: pd.DataFrame) -> np.ndarray:
        """Perform spectral clustering"""
        # Convert distance to affinity
        affinity = np.exp(-distance.values ** 2 / (2 * np.median(distance.values) ** 2))
        
        # Determine optimal number of clusters
        n_samples = distance.shape[0]
        n_clusters = min(10, max(2, n_samples // 5))
        
        spectral = SpectralClustering(
            n_clusters=n_clusters,
            affinity='precomputed',
            random_state=42
        )
        labels = spectral.fit_predict(affinity)
        
        return labels
    
    def kmeans_clustering(self, returns: pd.DataFrame) -> np.ndarray:
        """Perform K-means on PCA-reduced returns"""
        # Standardize returns
        returns_standardized = (returns - returns.mean()) / returns.std()
        
        # Apply PCA for dimensionality reduction
        pca = PCA(n_components=min(10, returns.shape[1]))
        returns_pca = pca.fit_transform(returns_standardized.T)
        
        # Determine optimal k using elbow method
        inertias = []
        max_k = min(15, returns_pca.shape[0])
        
        for k in range(2, max_k + 1):
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
            kmeans.fit(returns_pca)
            inertias.append(kmeans.inertia_)
        
        # Simple elbow detection (could be improved)
        diffs = np.diff(inertias)
        diffs_ratio = diffs[1:] / diffs[:-1]
        optimal_k = np.argmin(diffs_ratio) + 3  # +2 for diff indices, +1 for 0-index
        
        # Final clustering
        kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(returns_pca)
        
        self.cluster_metrics['pca_variance'] = pca.explained_variance_ratio_.sum()
        return labels
    
    def cluster_stocks(self, returns: pd.DataFrame) -> pd.DataFrame:
        """Main clustering function"""
        distance, corr_matrix = self.calculate_distance_matrix(returns)
        
        # Choose clustering method
        if self.config.CLUSTER_METHOD == 'hierarchical':
            labels = self.hierarchical_clustering(distance)
        elif self.config.CLUSTER_METHOD == 'optics':
            labels = self.OPTICS_clustering(distance)
        elif self.config.CLUSTER_METHOD == 'spectral':
            labels = self.spectral_clustering(distance)
        elif self.config.CLUSTER_METHOD == 'kmeans':
            labels = self.kmeans_clustering(returns)
        else:
            raise ValueError(f"Unknown clustering method: {self.config.CLUSTER_METHOD}")
        
        # Create cluster DataFrame
        cluster_df = pd.DataFrame({
            'ticker': returns.columns,
            'cluster': labels
        })
        
        # Filter clusters by size
        cluster_sizes = cluster_df['cluster'].value_counts()
        valid_clusters = cluster_sizes[
            (cluster_sizes >= self.config.MIN_CLUSTER_SIZE) & 
            (cluster_sizes <= self.config.MAX_CLUSTER_SIZE)
        ].index
        
        cluster_df = cluster_df[cluster_df['cluster'].isin(valid_clusters)]
        
        self.cluster_labels = cluster_df.set_index('ticker')['cluster']
        
        # Calculate cluster statistics
        self._calculate_cluster_stats(returns, cluster_df)
        
        return cluster_df
    
    def _calculate_cluster_stats(self, returns: pd.DataFrame, cluster_df: pd.DataFrame):
        """Calculate statistics for each cluster"""
        stats = {}
        for cluster_id in cluster_df['cluster'].unique():
            cluster_tickers = cluster_df[cluster_df['cluster'] == cluster_id]['ticker'].tolist()
            cluster_returns = returns[cluster_tickers]
            
            # Intra-cluster correlation
            intra_corr = cluster_returns.corr().values
            intra_corr_mean = intra_corr[np.triu_indices_from(intra_corr, k=1)].mean()
            
            stats[cluster_id] = {
                'size': len(cluster_tickers),
                'intra_correlation': intra_corr_mean,
                'tickers': cluster_tickers
            }
        
        self.cluster_metrics['cluster_stats'] = stats
    
    def visualize_clusters(self, returns: pd.DataFrame, cluster_df: pd.DataFrame):
        """Visualize clustering results"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. Correlation heatmap
        corr_matrix = returns.corr()
        sns.heatmap(corr_matrix, cmap='coolwarm', center=0, ax=axes[0, 0])
        axes[0, 0].set_title('Correlation Matrix')
        
        # 2. Distance matrix with cluster labels
        distance, _ = self.calculate_distance_matrix(returns)
        
        # Sort by cluster
        cluster_df_sorted = cluster_df.sort_values('cluster')
        distance_sorted = distance.loc[cluster_df_sorted['ticker'], cluster_df_sorted['ticker']]
        
        sns.heatmap(distance_sorted, cmap='viridis', ax=axes[0, 1])
        axes[0, 1].set_title('Distance Matrix (Sorted by Cluster)')
        
        # 3. Cluster sizes
        cluster_sizes = cluster_df['cluster'].value_counts().sort_index()
        axes[1, 0].bar(cluster_sizes.index.astype(str), cluster_sizes.values)
        axes[1, 0].set_xlabel('Cluster')
        axes[1, 0].set_ylabel('Number of Stocks')
        axes[1, 0].set_title('Cluster Sizes')
        
        # 4. Intra-cluster correlation distribution
        intra_corrs = []
        for cluster_id in self.cluster_metrics['cluster_stats']:
            intra_corrs.append(self.cluster_metrics['cluster_stats'][cluster_id]['intra_correlation'])
        
        axes[1, 1].hist(intra_corrs, bins=20, edgecolor='black')
        axes[1, 1].axvline(np.mean(intra_corrs), color='red', linestyle='--', label=f'Mean: {np.mean(intra_corrs):.3f}')
        axes[1, 1].set_xlabel('Intra-cluster Correlation')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].set_title('Distribution of Intra-cluster Correlations')
        axes[1, 1].legend()
        
        plt.tight_layout()
        plt.show()
        
        # Print cluster statistics
        print("\nCluster Statistics:")
        print("=" * 50)
        for cluster_id, stats in self.cluster_metrics['cluster_stats'].items():
            print(f"Cluster {cluster_id}:")
            print(f"  Size: {stats['size']} stocks")
            print(f"  Avg Intra-cluster Correlation: {stats['intra_correlation']:.3f}")
            print(f"  Tickers: {', '.join(stats['tickers'][:5])}{'...' if len(stats['tickers']) > 5 else ''}")
            print()


In [9]:

# %% [markdown]
# ## 6. Signal Generator

# %%
class SignalGenerator:
    """Generates trading signals based on mean-reversion within clusters"""
    
    def __init__(self, config):
        self.config = config
        self.signals = None
        self.signals_metadata = {}
        
    def calculate_zscore(self, series: pd.Series, lookback: int = 20) -> pd.Series:
        """Calculate rolling z-score"""
        rolling_mean = series.rolling(window=lookback).mean()
        rolling_std = series.rolling(window=lookback).std()
        return (series - rolling_mean) / rolling_std
    
    def generate_mean_reversion_signals(self, prices: pd.DataFrame, cluster_df: pd.DataFrame) -> pd.DataFrame:
        """Generate mean-reversion signals for each cluster"""
        signals = pd.DataFrame(index=prices.index, columns=prices.columns)
        signals_metadata = {}
        
        for cluster_id in cluster_df['cluster'].unique():
            cluster_tickers = cluster_df[cluster_df['cluster'] == cluster_id]['ticker'].tolist()
            cluster_prices = prices[cluster_tickers]
            
            # Calculate cluster mean (equally weighted)
            cluster_mean = cluster_prices.mean(axis=1)
            
            for ticker in cluster_tickers:
                # Calculate spread
                spread = cluster_prices[ticker] - cluster_mean
                
                # Calculate z-scores for multiple timeframes
                zscores = {}
                for window in self.config.LOOKBACK_WINDOWS:
                    zscores[f'z_{window}'] = self.calculate_zscore(spread, window)
                
                # Combined z-score (weighted average)
                weights = [0.4, 0.4, 0.2]  # Short, medium, long weights
                combined_z = sum(zscores[f'z_{window}'] * weight 
                                for window, weight in zip(self.config.LOOKBACK_WINDOWS, weights))
                
                # Generate signals with hysteresis
                long_entry = combined_z < self.config.ZSCORE_ENTRY_LONG
                long_exit = combined_z > self.config.ZSCORE_EXIT_LONG
                short_entry = combined_z > self.config.ZSCORE_ENTRY_SHORT
                short_exit = combined_z < self.config.ZSCORE_EXIT_SHORT
                
                # Initialize positions
                positions = pd.Series(0, index=combined_z.index)
                
                # Generate signals (vectorized for speed)
                positions = 0
                positions = np.where(long_entry, 1, positions)  # Long entry
                positions = np.where(short_entry, -1, positions)  # Short entry
                positions = np.where(long_exit & (positions == 1), 0, positions)  # Long exit
                positions = np.where(short_exit & (positions == -1), 0, positions)  # Short exit
                
                # Convert to Series and forward fill until exit signal
                positions = pd.Series(positions, index=combined_z.index)
                positions = positions.replace(to_replace=0, method='ffill').fillna(0)
                
                # Smooth positions to reduce whipsaw
                positions = positions.rolling(window=3, min_periods=1).mean()
                
                signals[ticker] = positions
                
                # Store metadata
                signals_metadata[ticker] = {
                    'cluster': cluster_id,
                    'avg_spread': spread.mean(),
                    'spread_std': spread.std(),
                    'avg_zscore': combined_z.mean(),
                    'signal_frequency': (positions.diff() != 0).mean()
                }
        
        self.signals = signals
        self.signals_metadata = signals_metadata
        
        return signals
    
    def apply_risk_filters(self, signals: pd.DataFrame, 
                          volatilities: pd.DataFrame,
                          volumes: pd.DataFrame) -> pd.DataFrame:
        """Apply risk management filters to signals"""
        filtered_signals = signals.copy()
        
        for ticker in signals.columns:
            signal_series = signals[ticker]
            
            # 1. Volume filter: only trade if volume > 20-day average
            if f"{ticker}_volume" in volumes.columns:
                volume_ma = volumes[f"{ticker}_volume"].rolling(20).mean()
                low_volume = volumes[f"{ticker}_volume"] < volume_ma * 0.5
                filtered_signals[ticker][low_volume] = 0
            
            # 2. Volatility filter: reduce position during high volatility
            if ticker in volatilities.columns:
                vol_rank = volatilities[ticker].rolling(60).rank(pct=True)
                high_vol = vol_rank > 0.8
                filtered_signals[ticker][high_vol] = filtered_signals[ticker][high_vol] * 0.5
            
            # 3. Trend filter: avoid trading against strong trend
            # Using ADX or simple moving average crossover
            if ticker in self.signals_metadata:
                prices = pd.DataFrame()  # Would need price data here
                # Add trend filter logic
        
        return filtered_signals
    
    def visualize_signals(self, ticker: str, prices: pd.Series, 
                         signals: pd.Series, cluster_mean: pd.Series = None):
        """Visualize signals for a specific ticker"""
        fig, axes = plt.subplots(3, 1, figsize=(15, 10), 
                                gridspec_kw={'height_ratios': [2, 1, 1]})
        
        # Plot 1: Price and signals
        axes[0].plot(prices.index, prices, label='Price', color='black', linewidth=1)
        
        # Highlight long positions
        long_positions = signals == 1
        if long_positions.any():
            axes[0].fill_between(prices.index, prices.min(), prices.max(),
                                where=long_positions, alpha=0.3, color='green', 
                                label='Long Position')
        
        # Highlight short positions
        short_positions = signals == -1
        if short_positions.any():
            axes[0].fill_between(prices.index, prices.min(), prices.max(),
                                where=short_positions, alpha=0.3, color='red', 
                                label='Short Position')
        
        if cluster_mean is not None:
            axes[0].plot(cluster_mean.index, cluster_mean, 
                        label='Cluster Mean', color='blue', linestyle='--')
        
        axes[0].set_title(f'{ticker} - Trading Signals')
        axes[0].set_ylabel('Price')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Plot 2: Spread and z-score
        if cluster_mean is not None:
            spread = prices - cluster_mean
            zscore = (spread - spread.rolling(20).mean()) / spread.rolling(20).std()
            
            axes[1].plot(spread.index, spread, label='Spread', color='purple')
            axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
            axes[1].fill_between(spread.index, -spread.std(), spread.std(),
                                alpha=0.1, color='gray')
            axes[1].set_ylabel('Spread')
            axes[1].legend()
            axes[1].grid(True, alpha=0.3)
            
            axes[2].plot(zscore.index, zscore, label='Z-Score', color='orange')
            axes[2].axhline(y=self.config.ZSCORE_ENTRY_LONG, color='green', 
                           linestyle='--', label=f'Long Entry ({self.config.ZSCORE_ENTRY_LONG})')
            axes[2].axhline(y=self.config.ZSCORE_EXIT_LONG, color='green', 
                           linestyle=':', label=f'Long Exit ({self.config.ZSCORE_EXIT_LONG})')
            axes[2].axhline(y=self.config.ZSCORE_ENTRY_SHORT, color='red', 
                           linestyle='--', label=f'Short Entry ({self.config.ZSCORE_ENTRY_SHORT})')
            axes[2].axhline(y=self.config.ZSCORE_EXIT_SHORT, color='red', 
                           linestyle=':', label=f'Short Exit ({self.config.ZSCORE_EXIT_SHORT})')
            axes[2].axhline(y=0, color='black', linestyle='-', alpha=0.3)
            axes[2].set_ylabel('Z-Score')
            axes[2].set_xlabel('Date')
            axes[2].legend()
            axes[2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print signal statistics
        if ticker in self.signals_metadata:
            metadata = self.signals_metadata[ticker]
            print(f"\nSignal Statistics for {ticker}:")
            print(f"Cluster: {metadata['cluster']}")
            print(f"Average Spread: {metadata['avg_spread']:.2f}")
            print(f"Spread Std: {metadata['spread_std']:.2f}")
            print(f"Average Z-Score: {metadata['avg_zscore']:.2f}")
            print(f"Signal Frequency: {metadata['signal_frequency']:.2%}")


In [10]:
# %% [markdown]
# ## 7. Risk Manager & Position Sizer

# %%
class RiskManager:
    """Manages risk and calculates position sizes"""
    
    def __init__(self, config):
        self.config = config
        self.positions = None
        self.risk_metrics = {}
        
    def calculate_position_sizes(self, signals: pd.DataFrame, 
                                returns: pd.DataFrame,
                                volatilities: pd.DataFrame,
                                capital: float = None) -> pd.DataFrame:
        """Calculate position sizes using Kelly Criterion and volatility scaling"""
        if capital is None:
            capital = self.config.INITIAL_CAPITAL
        
        positions = pd.DataFrame(index=signals.index, columns=signals.columns)
        
        for ticker in signals.columns:
            signal = signals[ticker]
            
            # Base position from signal
            base_position = signal.copy()
            
            # 1. Volatility scaling
            if ticker in volatilities.columns:
                # Calculate realized volatility
                realized_vol = returns[ticker].rolling(20).std() * np.sqrt(252)
                realized_vol = realized_vol.replace(0, np.nan).ffill().fillna(0.3)
                
                # Volatility scaling factor
                vol_scaling = self.config.TARGET_VOLATILITY / realized_vol
                vol_scaling = vol_scaling.clip(0.5, 2.0)  # Limit scaling
            else:
                vol_scaling = 1.0
            
            # 2. Kelly position sizing (simplified)
            # Estimate win rate and win/loss ratio from historical signals
            # For simplicity, using fixed values - could be estimated from backtest
            win_rate = 0.55  # Estimated from strategy logic
            avg_win_loss_ratio = 1.2  # Estimated
            
            kelly_f = win_rate - (1 - win_rate) / avg_win_loss_ratio
            kelly_f = max(0.01, min(kelly_f, 0.25))  # Cap Kelly fraction
            
            # 3. Combine factors
            position_size = base_position * kelly_f * vol_scaling
            
            # 4. Apply position limits
            position_size = position_size.clip(-self.config.MAX_POSITION_SIZE, 
                                              self.config.MAX_POSITION_SIZE)
            
            positions[ticker] = position_size
        
        # 5. Portfolio-level constraints
        # Gross exposure limit
        gross_exposure = positions.abs().sum(axis=1)
        scaling_factor = np.minimum(1, self.config.MAX_PORTFOLIO_EXPOSURE / gross_exposure)
        scaling_factor = scaling_factor.replace([np.inf, -np.inf], 1)
        
        positions = positions.mul(scaling_factor, axis=0)
        
        # Net exposure limit (optional)
        # net_exposure = positions.sum(axis=1)
        # Could add net exposure constraint here
        
        self.positions = positions
        return positions
    
    def apply_stop_loss(self, positions: pd.DataFrame, 
                       prices: pd.DataFrame,
                       initial_stop: float = 0.05,
                       trailing_stop: float = 0.03) -> pd.DataFrame:
        """Apply stop-loss logic to positions"""
        positions_with_stop = positions.copy()
        
        for ticker in positions.columns:
            position_series = positions[ticker]
            price_series = prices[ticker]
            
            # Initialize tracking variables
            entry_price = None
            highest_price = None
            lowest_price = None
            
            for i in range(1, len(position_series)):
                current_position = position_series.iloc[i]
                prev_position = position_series.iloc[i-1]
                current_price = price_series.iloc[i]
                
                # Entry signal
                if current_position != 0 and prev_position == 0:
                    entry_price = current_price
                    highest_price = current_price if current_position > 0 else None
                    lowest_price = current_price if current_position < 0 else None
                
                # Exit signal
                elif current_position == 0 and prev_position != 0:
                    entry_price = None
                    highest_price = None
                    lowest_price = None
                
                # Check stop-loss for long positions
                elif current_position > 0 and entry_price is not None:
                    highest_price = max(highest_price, current_price)
                    
                    # Trailing stop
                    if current_price < highest_price * (1 - trailing_stop):
                        positions_with_stop.iloc[i, positions.columns.get_loc(ticker)] = 0
                        entry_price = None
                        highest_price = None
                    
                    # Initial stop
                    elif current_price < entry_price * (1 - initial_stop):
                        positions_with_stop.iloc[i, positions.columns.get_loc(ticker)] = 0
                        entry_price = None
                        highest_price = None
                
                # Check stop-loss for short positions
                elif current_position < 0 and entry_price is not None:
                    lowest_price = min(lowest_price, current_price)
                    
                    # Trailing stop
                    if current_price > lowest_price * (1 + trailing_stop):
                        positions_with_stop.iloc[i, positions.columns.get_loc(ticker)] = 0
                        entry_price = None
                        lowest_price = None
                    
                    # Initial stop
                    elif current_price > entry_price * (1 + initial_stop):
                        positions_with_stop.iloc[i, positions.columns.get_loc(ticker)] = 0
                        entry_price = None
                        lowest_price = None
        
        return positions_with_stop
    
    def calculate_risk_metrics(self, positions: pd.DataFrame, 
                              returns: pd.DataFrame) -> Dict:
        """Calculate risk metrics for the portfolio"""
        # Portfolio returns
        portfolio_returns = (positions.shift(1) * returns).sum(axis=1)
        
        # Basic metrics
        metrics = {
            'total_return': portfolio_returns.sum(),
            'annual_return': portfolio_returns.mean() * 252,
            'annual_volatility': portfolio_returns.std() * np.sqrt(252),
            'sharpe_ratio': (portfolio_returns.mean() / portfolio_returns.std()) * np.sqrt(252) 
                           if portfolio_returns.std() > 0 else 0,
            'max_drawdown': self._calculate_max_drawdown(portfolio_returns),
            'win_rate': (portfolio_returns > 0).mean(),
            'profit_factor': abs(portfolio_returns[portfolio_returns > 0].sum() / 
                               portfolio_returns[portfolio_returns < 0].sum()) 
                           if portfolio_returns[portfolio_returns < 0].sum() != 0 else np.inf
        }
        
        # Position metrics
        metrics['avg_position_count'] = positions.abs().gt(0).sum(axis=1).mean()
        metrics['avg_gross_exposure'] = positions.abs().sum(axis=1).mean()
        metrics['avg_net_exposure'] = positions.sum(axis=1).abs().mean()
        
        self.risk_metrics = metrics
        return metrics
    
    def _calculate_max_drawdown(self, returns: pd.Series) -> float:
        """Calculate maximum drawdown"""
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        return drawdown.min()
    
    def visualize_risk_metrics(self, positions: pd.DataFrame, returns: pd.DataFrame):
        """Visualize risk metrics and positions"""
        portfolio_returns = (positions.shift(1) * returns).sum(axis=1)
        cumulative_returns = (1 + portfolio_returns).cumprod()
        
        fig, axes = plt.subplots(3, 2, figsize=(15, 12))
        
        # Plot 1: Cumulative returns
        axes[0, 0].plot(cumulative_returns.index, cumulative_returns)
        axes[0, 0].set_title('Cumulative Portfolio Returns')
        axes[0, 0].set_ylabel('Cumulative Return')
        axes[0, 0].grid(True, alpha=0.3)
        
        # Plot 2: Drawdown
        running_max = cumulative_returns.expanding().max()
        drawdown = (cumulative_returns - running_max) / running_max
        axes[0, 1].fill_between(drawdown.index, 0, drawdown.values, alpha=0.5, color='red')
        axes[0, 1].set_title('Portfolio Drawdown')
        axes[0, 1].set_ylabel('Drawdown')
        axes[0, 1].grid(True, alpha=0.3)
        
        # Plot 3: Position count
        position_count = positions.abs().gt(0).sum(axis=1)
        axes[1, 0].plot(position_count.index, position_count)
        axes[1, 0].set_title('Number of Active Positions')
        axes[1, 0].set_ylabel('Count')
        axes[1, 0].grid(True, alpha=0.3)
        
        # Plot 4: Gross exposure
        gross_exposure = positions.abs().sum(axis=1)
        axes[1, 1].plot(gross_exposure.index, gross_exposure)
        axes[1, 1].axhline(y=self.config.MAX_PORTFOLIO_EXPOSURE, color='red', 
                          linestyle='--', label='Max Exposure')
        axes[1, 1].set_title('Gross Exposure')
        axes[1, 1].set_ylabel('Exposure')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        # Plot 5: Monthly returns
        monthly_returns = portfolio_returns.resample('M').sum()
        axes[2, 0].bar(monthly_returns.index.strftime('%Y-%m'), monthly_returns.values)
        axes[2, 0].axhline(y=0, color='black', linewidth=0.5)
        axes[2, 0].set_title('Monthly Returns')
        axes[2, 0].set_ylabel('Return')
        axes[2, 0].tick_params(axis='x', rotation=45)
        axes[2, 0].grid(True, alpha=0.3)
        
        # Plot 6: Return distribution
        axes[2, 1].hist(portfolio_returns, bins=50, edgecolor='black', alpha=0.7)
        axes[2, 1].axvline(x=portfolio_returns.mean(), color='red', 
                          linestyle='--', label=f'Mean: {portfolio_returns.mean():.4f}')
        axes[2, 1].set_title('Daily Return Distribution')
        axes[2, 1].set_xlabel('Return')
        axes[2, 1].set_ylabel('Frequency')
        axes[2, 1].legend()
        axes[2, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print risk metrics
        print("\nPortfolio Risk Metrics:")
        print("=" * 50)
        for metric, value in self.risk_metrics.items():
            if isinstance(value, float):
                if 'rate' in metric or 'factor' in metric:
                    print(f"{metric.replace('_', ' ').title()}: {value:.3f}")
                elif 'return' in metric or 'ratio' in metric or 'drawdown' in metric:
                    print(f"{metric.replace('_', ' ').title()}: {value:.3%}")
                else:
                    print(f"{metric.replace('_', ' ').title()}: {value:.3f}")
            else:
                print(f"{metric.replace('_', ' ').title()}: {value}")


In [11]:

# %% [markdown]
# ## 8. Backtesting Engine

# %%
class BacktestingEngine:
    """Comprehensive backtesting engine"""
    
    def __init__(self, config):
        self.config = config
        self.results = {}
        self.performance_report = {}
        
    def run_backtest(self, prices: pd.DataFrame, 
                    positions: pd.DataFrame,
                    transaction_cost: float = None) -> Dict:
        """Run backtest with transaction costs"""
        if transaction_cost is None:
            transaction_cost = self.config.TRANSACTION_COST
        
        # Calculate returns
        returns = prices.pct_change()
        
        # Initialize portfolio
        portfolio_value = pd.Series(index=prices.index, dtype=float)
        portfolio_value.iloc[0] = self.config.INITIAL_CAPITAL
        
        # Track trades and costs
        trades = []
        costs = pd.Series(0, index=prices.index)
        holdings = pd.DataFrame(0, index=prices.index, columns=prices.columns)
        
        # Simulate trading
        for i in range(1, len(prices)):
            prev_date = prices.index[i-1]
            current_date = prices.index[i]
            
            # Previous holdings
            prev_holdings = holdings.iloc[i-1].copy()
            
            # Target holdings based on positions
            target_weights = positions.loc[current_date]
            target_value = portfolio_value.iloc[i-1] * target_weights
            
            # Calculate trades
            trade_values = target_value - prev_holdings * prices.loc[current_date]
            
            # Calculate transaction costs
            daily_cost = abs(trade_values).sum() * transaction_cost
            costs.loc[current_date] = daily_cost
            
            # Update holdings
            holdings.loc[current_date] = target_weights * portfolio_value.iloc[i-1] / prices.loc[current_date]
            
            # Calculate portfolio value
            portfolio_value.loc[current_date] = (holdings.loc[current_date] * prices.loc[current_date]).sum()
            
            # Deduct transaction costs
            portfolio_value.loc[current_date] -= daily_cost
            
            # Record trades
            if trade_values.abs().sum() > 0:
                trades.append({
                    'date': current_date,
                    'trade_value': trade_values.sum(),
                    'cost': daily_cost
                })
        
        # Calculate portfolio returns
        portfolio_returns = portfolio_value.pct_change().dropna()
        
        # Store results
        self.results = {
            'portfolio_value': portfolio_value,
            'portfolio_returns': portfolio_returns,
            'holdings': holdings,
            'trades': trades,
            'costs': costs,
            'positions': positions
        }
        
        return self.results
    
    def calculate_performance_metrics(self) -> Dict:
        """Calculate comprehensive performance metrics"""
        if not self.results:
            raise ValueError("Run backtest first")
        
        returns = self.results['portfolio_returns']
        portfolio_value = self.results['portfolio_value']
        
        # Basic metrics
        total_return = portfolio_value.iloc[-1] / portfolio_value.iloc[0] - 1
        annual_return = (1 + total_return) ** (252 / len(returns)) - 1
        
        # Risk metrics
        annual_volatility = returns.std() * np.sqrt(252)
        sharpe_ratio = annual_return / annual_volatility if annual_volatility > 0 else 0
        
        # Drawdown
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_drawdown = drawdown.min()
        
        # Sortino ratio (downside deviation)
        downside_returns = returns[returns < 0]
        downside_deviation = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 0 else 0
        sortino_ratio = annual_return / downside_deviation if downside_deviation > 0 else 0
        
        # Calmar ratio
        calmar_ratio = annual_return / abs(max_drawdown) if max_drawdown != 0 else 0
        
        # Win rate and profit factor
        win_rate = (returns > 0).mean()
        gross_profit = returns[returns > 0].sum()
        gross_loss = abs(returns[returns < 0].sum())
        profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf
        
        # Trade statistics
        trades = self.results['trades']
        n_trades = len(trades)
        avg_trade_return = returns.mean()
        
        # Transaction costs
        total_costs = self.results['costs'].sum()
        cost_per_trade = total_costs / n_trades if n_trades > 0 else 0
        
        # Store metrics
        self.performance_report = {
            'total_return': total_return,
            'annual_return': annual_return,
            'annual_volatility': annual_volatility,
            'sharpe_ratio': sharpe_ratio,
            'sortino_ratio': sortino_ratio,
            'calmar_ratio': calmar_ratio,
            'max_drawdown': max_drawdown,
            'win_rate': win_rate,
            'profit_factor': profit_factor,
            'n_trades': n_trades,
            'avg_trade_return': avg_trade_return,
            'total_costs': total_costs,
            'cost_per_trade': cost_per_trade,
            'final_portfolio_value': portfolio_value.iloc[-1]
        }
        
        return self.performance_report
    
    def generate_report(self):
        """Generate comprehensive performance report"""
        if not self.performance_report:
            self.calculate_performance_metrics()
        
        report = """
        PERFORMANCE REPORT
        ===================
        
        Returns:
        --------
        Total Return: {total_return:.2%}
        Annual Return: {annual_return:.2%}
        Annual Volatility: {annual_volatility:.2%}
        
        Risk-Adjusted Returns:
        ----------------------
        Sharpe Ratio: {sharpe_ratio:.2f}
        Sortino Ratio: {sortino_ratio:.2f}
        Calmar Ratio: {calmar_ratio:.2f}
        
        Risk Metrics:
        -------------
        Maximum Drawdown: {max_drawdown:.2%}
        Win Rate: {win_rate:.2%}
        Profit Factor: {profit_factor:.2f}
        
        Trade Statistics:
        -----------------
        Number of Trades: {n_trades:,}
        Average Trade Return: {avg_trade_return:.4%}
        Total Transaction Costs: ${total_costs:,.2f}
        Cost per Trade: ${cost_per_trade:,.2f}
        
        Final Portfolio Value: ${final_portfolio_value:,.2f}
        """
        
        print(report.format(**self.performance_report))
    
    def plot_results(self):
        """Plot backtest results"""
        if not self.results:
            raise ValueError("Run backtest first")
        
        fig, axes = plt.subplots(3, 2, figsize=(15, 12))
        
        # Plot 1: Portfolio value
        axes[0, 0].plot(self.results['portfolio_value'])
        axes[0, 0].set_title('Portfolio Value')
        axes[0, 0].set_ylabel('Value ($)')
        axes[0, 0].grid(True, alpha=0.3)
        
        # Plot 2: Cumulative returns
        cumulative_returns = (1 + self.results['portfolio_returns']).cumprod()
        axes[0, 1].plot(cumulative_returns)
        axes[0, 1].set_title('Cumulative Returns')
        axes[0, 1].set_ylabel('Cumulative Return')
        axes[0, 1].grid(True, alpha=0.3)
        
        # Plot 3: Drawdown
        running_max = cumulative_returns.expanding().max()
        drawdown = (cumulative_returns - running_max) / running_max
        axes[1, 0].fill_between(drawdown.index, 0, drawdown.values, alpha=0.5, color='red')
        axes[1, 0].set_title('Portfolio Drawdown')
        axes[1, 0].set_ylabel('Drawdown')
        axes[1, 0].grid(True, alpha=0.3)
        
        # Plot 4: Monthly returns
        monthly_returns = self.results['portfolio_returns'].resample('M').sum()
        axes[1, 1].bar(monthly_returns.index.strftime('%Y-%m'), monthly_returns.values)
        axes[1, 1].axhline(y=0, color='black', linewidth=0.5)
        axes[1, 1].set_title('Monthly Returns')
        axes[1, 1].set_ylabel('Return')
        axes[1, 1].tick_params(axis='x', rotation=45)
        axes[1, 1].grid(True, alpha=0.3)
        
        # Plot 5: Position heatmap (simplified)
        position_data = self.results['positions'].resample('W').last()
        if len(position_data.columns) > 20:
            # Show only top 20 positions by frequency
            position_freq = (self.results['positions'] != 0).sum()
            top_tickers = position_freq.nlargest(20).index
            position_data = position_data[top_tickers]
        
        im = axes[2, 0].imshow(position_data.T, aspect='auto', cmap='coolwarm')
        axes[2, 0].set_title('Position Heatmap (Weekly)')
        axes[2, 0].set_xlabel('Time')
        axes[2, 0].set_ylabel('Ticker')
        plt.colorbar(im, ax=axes[2, 0])
        
        # Plot 6: Return distribution
        axes[2, 1].hist(self.results['portfolio_returns'], bins=50, edgecolor='black', alpha=0.7)
        axes[2, 1].axvline(x=self.results['portfolio_returns'].mean(), color='red', 
                          linestyle='--', label=f'Mean: {self.results['portfolio_returns'].mean():.4f}')
        axes[2, 1].set_title('Daily Return Distribution')
        axes[2, 1].set_xlabel('Return')
        axes[2, 1].set_ylabel('Frequency')
        axes[2, 1].legend()
        axes[2, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

In [12]:
# %% [markdown]
# ## 9. Walk-Forward Optimization

# %%
class WalkForwardOptimizer:
    """Perform walk-forward optimization to avoid overfitting"""
    
    def __init__(self, config):
        self.config = config
        self.optimization_results = {}
        
    def optimize_parameters(self, data: Dict, 
                           train_start: str,
                           train_end: str) -> Dict:
        """Optimize parameters on training data"""
        # Extract training data
        mask = (data['prices'].index >= train_start) & (data['prices'].index <= train_end)
        train_data = {
            'prices': data['prices'][mask],
            'returns': data['returns'][mask],
            'volatilities': data['volatilities'][mask]
        }
        
        # Define parameter grid
        param_grid = {
            'ZSCORE_ENTRY_LONG': [-1.0, -1.5, -2.0],
            'ZSCORE_ENTRY_SHORT': [1.0, 1.5, 2.0],
            'ZSCORE_EXIT_LONG': [-0.25, -0.5, 0],
            'ZSCORE_EXIT_SHORT': [0.25, 0.5, 0],
            'LOOKBACK_WINDOWS': [[10, 20, 60], [5, 15, 30], [20, 40, 80]]
        }
        
        best_score = -np.inf
        best_params = None
        
        # Simple grid search (could be enhanced with Bayesian optimization)
        # Note: This is simplified for demonstration
        for entry_long in param_grid['ZSCORE_ENTRY_LONG']:
            for entry_short in param_grid['ZSCORE_ENTRY_SHORT']:
                # Update config
                temp_config = Config()
                temp_config.ZSCORE_ENTRY_LONG = entry_long
                temp_config.ZSCORE_ENTRY_SHORT = entry_short
                
                try:
                    # Test strategy with these parameters
                    clusterer = StockClustering(temp_config)
                    cluster_df = clusterer.cluster_stocks(train_data['returns'])
                    
                    signal_gen = SignalGenerator(temp_config)
                    signals = signal_gen.generate_mean_reversion_signals(
                        train_data['prices'], cluster_df
                    )
                    
                    risk_manager = RiskManager(temp_config)
                    positions = risk_manager.calculate_position_sizes(
                        signals, train_data['returns'], train_data['volatilities']
                    )
                    
                    # Calculate performance
                    backtester = BacktestingEngine(temp_config)
                    backtester.run_backtest(train_data['prices'], positions)
                    metrics = backtester.calculate_performance_metrics()
                    
                    # Score based on Sharpe ratio
                    score = metrics['sharpe_ratio']
                    
                    if score > best_score:
                        best_score = score
                        best_params = {
                            'ZSCORE_ENTRY_LONG': entry_long,
                            'ZSCORE_ENTRY_SHORT': entry_short,
                            'score': score
                        }
                except:
                    continue
        
        return best_params
    
    def walk_forward_test(self, data: Dict, 
                         train_months: int = 24,
                         test_months: int = 6) -> pd.DataFrame:
        """Perform walk-forward testing"""
        results = []
        dates = data['prices'].index
        
        current_date = dates[0]
        while True:
            # Calculate train and test periods
            train_end = current_date + pd.DateOffset(months=train_months)
            test_end = train_end + pd.DateOffset(months=test_months)
            
            if test_end > dates[-1]:
                break
            
            print(f"\nPeriod: Train {current_date.date()} to {train_end.date()}, "
                  f"Test {train_end.date()} to {test_end.date()}")
            
            # Optimize on training data
            optimal_params = self.optimize_parameters(data, current_date, train_end)
            
            if optimal_params is None:
                current_date = train_end
                continue
            
            # Test on out-of-sample data
            test_mask = (dates >= train_end) & (dates <= test_end)
            test_data = {
                'prices': data['prices'][test_mask],
                'returns': data['returns'][test_mask],
                'volatilities': data['volatilities'][test_mask]
            }
            
            # Apply optimal parameters
            test_config = Config()
            test_config.ZSCORE_ENTRY_LONG = optimal_params['ZSCORE_ENTRY_LONG']
            test_config.ZSCORE_ENTRY_SHORT = optimal_params['ZSCORE_ENTRY_SHORT']
            
            # Run strategy on test data
            clusterer = StockClustering(test_config)
            cluster_df = clusterer.cluster_stocks(test_data['returns'])
            
            signal_gen = SignalGenerator(test_config)
            signals = signal_gen.generate_mean_reversion_signals(
                test_data['prices'], cluster_df
            )
            
            risk_manager = RiskManager(test_config)
            positions = risk_manager.calculate_position_sizes(
                signals, test_data['returns'], test_data['volatilities']
            )
            
            backtester = BacktestingEngine(test_config)
            backtester.run_backtest(test_data['prices'], positions)
            test_metrics = backtester.calculate_performance_metrics()
            
            # Store results
            period_result = {
                'train_start': current_date,
                'train_end': train_end,
                'test_start': train_end,
                'test_end': test_end,
                'optimal_params': optimal_params,
                'test_metrics': test_metrics
            }
            results.append(period_result)
            
            # Move forward
            current_date = train_end
        
        # Aggregate results
        results_df = pd.DataFrame([{
            'period': f"{r['test_start'].date()} to {r['test_end'].date()}",
            'sharpe': r['test_metrics']['sharpe_ratio'],
            'return': r['test_metrics']['annual_return'],
            'max_dd': r['test_metrics']['max_drawdown'],
            'win_rate': r['test_metrics']['win_rate']
        } for r in results])
        
        self.optimization_results = results_df
        return results_df
    
    def visualize_walk_forward(self):
        """Visualize walk-forward optimization results"""
        if self.optimization_results.empty:
            raise ValueError("Run walk-forward test first")
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Plot 1: Sharpe ratio over time
        axes[0, 0].bar(range(len(self.optimization_results)), 
                      self.optimization_results['sharpe'])
        axes[0, 0].axhline(y=self.optimization_results['sharpe'].mean(), 
                          color='red', linestyle='--', 
                          label=f'Mean: {self.optimization_results["sharpe"].mean():.2f}')
        axes[0, 0].set_title('Sharpe Ratio by Period')
        axes[0, 0].set_xlabel('Period')
        axes[0, 0].set_ylabel('Sharpe Ratio')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # Plot 2: Annual return over time
        axes[0, 1].bar(range(len(self.optimization_results)), 
                      self.optimization_results['return'])
        axes[0, 1].axhline(y=self.optimization_results['return'].mean(), 
                          color='red', linestyle='--', 
                          label=f'Mean: {self.optimization_results["return"].mean():.2%}')
        axes[0, 1].set_title('Annual Return by Period')
        axes[0, 1].set_xlabel('Period')
        axes[0, 1].set_ylabel('Annual Return')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # Plot 3: Max drawdown over time
        axes[1, 0].bar(range(len(self.optimization_results)), 
                      self.optimization_results['max_dd'])
        axes[1, 0].axhline(y=self.optimization_results['max_dd'].mean(), 
                          color='red', linestyle='--', 
                          label=f'Mean: {self.optimization_results["max_dd"].mean():.2%}')
        axes[1, 0].set_title('Max Drawdown by Period')
        axes[1, 0].set_xlabel('Period')
        axes[1, 0].set_ylabel('Max Drawdown')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # Plot 4: Win rate over time
        axes[1, 1].bar(range(len(self.optimization_results)), 
                      self.optimization_results['win_rate'])
        axes[1, 1].axhline(y=self.optimization_results['win_rate'].mean(), 
                          color='red', linestyle='--', 
                          label=f'Mean: {self.optimization_results["win_rate"].mean():.2%}')
        axes[1, 1].set_title('Win Rate by Period')
        axes[1, 1].set_xlabel('Period')
        axes[1, 1].set_ylabel('Win Rate')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print summary statistics
        print("\nWalk-Forward Optimization Summary:")
        print("=" * 50)
        for metric in ['sharpe', 'return', 'max_dd', 'win_rate']:
            mean_val = self.optimization_results[metric].mean()
            std_val = self.optimization_results[metric].std()
            if metric in ['return', 'max_dd', 'win_rate']:
                print(f"{metric.replace('_', ' ').title()}: {mean_val:.2%} (±{std_val:.2%})")
            else:
                print(f"{metric.replace('_', ' ').title()}: {mean_val:.2f} (±{std_val:.2f})")


In [13]:

# %% [markdown]
# ## 10. Main Execution Pipeline

# %%
class TradingStrategyPipeline:
    """Main pipeline to execute the complete trading strategy"""
    
    def __init__(self, config):
        self.config = config
        self.data = None
        self.cluster_df = None
        self.signals = None
        self.positions = None
        self.backtest_results = None
        self.performance_report = None
        
    def run(self, tickers: List[str] = None, walk_forward: bool = False):
        """Run the complete trading strategy pipeline"""
        print("=" * 60)
        print("STOCK CLUSTERING & MEAN-REVERSION TRADING STRATEGY")
        print("=" * 60)
        
        # Step 1: Fetch and preprocess data
        print("\n1. Fetching and preprocessing data...")
        data_fetcher = DataFetcher(self.config)
        self.data = data_fetcher.fetch_data(tickers)
        self.data = data_fetcher.preprocess_data()
        
        # Step 2: Cluster stocks
        print("\n2. Clustering stocks...")
        clusterer = StockClustering(self.config)
        self.cluster_df = clusterer.cluster_stocks(self.data['returns'])
        clusterer.visualize_clusters(self.data['returns'], self.cluster_df)
        
        if walk_forward:
            # Step 3A: Walk-forward optimization
            print("\n3. Performing walk-forward optimization...")
            wf_optimizer = WalkForwardOptimizer(self.config)
            wf_results = wf_optimizer.walk_forward_test(self.data)
            wf_optimizer.visualize_walk_forward()
            
            # Use average optimal parameters from walk-forward
            avg_entry_long = -1.0  # Would come from optimization
            avg_entry_short = 1.0  # Would come from optimization
            
            self.config.ZSCORE_ENTRY_LONG = avg_entry_long
            self.config.ZSCORE_ENTRY_SHORT = avg_entry_short
            
        # Step 3B: Generate signals
        print("\n3. Generating trading signals...")
        signal_gen = SignalGenerator(self.config)
        self.signals = signal_gen.generate_mean_reversion_signals(
            self.data['prices'], self.cluster_df
        )
        
        # Visualize signals for a sample ticker
        sample_ticker = self.cluster_df.iloc[0]['ticker']
        cluster_id = self.cluster_df.iloc[0]['cluster']
        cluster_tickers = self.cluster_df[self.cluster_df['cluster'] == cluster_id]['ticker'].tolist()
        cluster_mean = self.data['prices'][cluster_tickers].mean(axis=1)
        
        signal_gen.visualize_signals(
            sample_ticker,
            self.data['prices'][sample_ticker],
            self.signals[sample_ticker],
            cluster_mean
        )
        
        # Step 4: Risk management and position sizing
        print("\n4. Calculating position sizes with risk management...")
        risk_manager = RiskManager(self.config)
        self.positions = risk_manager.calculate_position_sizes(
            self.signals, 
            self.data['returns'], 
            self.data['volatilities']
        )
        
        # Apply stop-loss
        self.positions = risk_manager.apply_stop_loss(
            self.positions, 
            self.data['prices']
        )
        
        # Calculate risk metrics
        risk_metrics = risk_manager.calculate_risk_metrics(
            self.positions, 
            self.data['returns']
        )
        risk_manager.visualize_risk_metrics(self.positions, self.data['returns'])
        
        # Step 5: Backtesting
        print("\n5. Running backtest...")
        backtester = BacktestingEngine(self.config)
        self.backtest_results = backtester.run_backtest(
            self.data['prices'], 
            self.positions
        )
        
        # Step 6: Performance analysis
        print("\n6. Analyzing performance...")
        self.performance_report = backtester.calculate_performance_metrics()
        backtester.generate_report()
        backtester.plot_results()
        
        print("\nPipeline execution complete!")
        
        return {
            'data': self.data,
            'cluster_df': self.cluster_df,
            'signals': self.signals,
            'positions': self.positions,
            'backtest_results': self.backtest_results,
            'performance_report': self.performance_report
        }
    
    def export_results(self, filename: str = 'trading_strategy_results'):
        """Export all results to files"""
        import pickle
        
        results = {
            'config': self.config.__dict__,
            'data': self.data,
            'cluster_df': self.cluster_df,
            'signals': self.signals,
            'positions': self.positions,
            'backtest_results': self.backtest_results,
            'performance_report': self.performance_report
        }
        
        # Save as pickle
        with open(f'{filename}.pkl', 'wb') as f:
            pickle.dump(results, f)
        
        # Save cluster assignments as CSV
        self.cluster_df.to_csv(f'{filename}_clusters.csv')
        
        # Save performance report as JSON
        with open(f'{filename}_performance.json', 'w') as f:
            json.dump(self.performance_report, f, indent=2, default=str)
        
        print(f"\nResults exported to {filename}.*")


In [14]:
# %% [markdown]
# ## 11. Execute the Strategy

# %%
if __name__ == "__main__":
    # Initialize configuration
    config = Config()
    
    # Optionally modify config here
    # config.TICKER_LIMIT = 100
    # config.CLUSTER_METHOD = 'spectral'
    # config.ZSCORE_ENTRY_LONG = -1.5
    
    # Create and run pipeline
    pipeline = TradingStrategyPipeline(config)
    
    # Run with walk-forward optimization (set to False for faster execution)
    results = pipeline.run(walk_forward=False)
    
    # Export results
    pipeline.export_results('strategy_results')

STOCK CLUSTERING & MEAN-REVERSION TRADING STRATEGY

1. Fetching and preprocessing data...


INFO:__main__:Connected to MongoDB successfully
INFO:__main__:Fetched 50 tickers


ValueError: No valid data fetched